# Paper 4 – Experiments: calibration-guided robust reallocation (Kerala 2018 and 2019)

This notebook runs the complete evaluation, writing the tables and figures the paper needs:

| Output | Content |
|---|---|
| Table 3 | Forecasting skill: accuracy, macro precision, recall and F1, ROC-AUC |
| Table 4 | Forecast calibration (ECE, Brier, NLL) before and after temperature scaling; demand-estimation MAE and RMSE |
| Table 5 | Allocation comparison: B1 static, B2 proportional, B3 stochastic, B4 fixed robust, B5 learned, **FU** forecast-based (uncalibrated), **FC** calibration-guided (nominal), **CGRR** proposed. 200 runs each; Wilcoxon tests with Holm correction |
| Table 6 | Robustness: stress S1–S4 (S4 is sudden escalation), α × η sensitivity, alternative demand assumptions |
| Table 7 | Ablations A1 (no conformal sets), A2a/A2b (re-plan every period / never), A3 (no inundation or landslide routing) |
| Figures | Reliability diagram, coverage vs α, trigger timeline (600 dpi TIFF) |
| `run_log.txt` | Proof of execution: timestamps, package versions, configuration and input checksums |

**Run once with `SMOKE_TEST = True`** to check the code on random data. **Never put those numbers in the paper.** Then switch to `False` for the real Kerala inputs.

Realized demand is a **modelled proxy**: observed exposed population multiplied by a need share. It is not observed emergency demand, and the paper says so. Table 6c tests alternative demand assumptions.

In [ ]:
!pip -q install "pulp==2.9.0" highspy osmnx geopandas pyyaml

In [ ]:
SMOKE_TEST = False
EVENTS = ['2018', '2019']
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/Paper4_Kerala'     # notebook-01 outputs + forecasts, calibration, stocks
RESULTS = DATA + '/results'
import os; os.makedirs('/content/src', exist_ok=True)

## 0. Final-run inputs
Writes `stocks_2018.csv`, `stocks_2019.csv`, `landslides_2019.csv` and downloads the 2018 landslide inventory (Hao et al. 2020) into `Paper4_Kerala/`. Stops with an error if a hub name does not match a district origin.

In [ ]:
# Final-run inputs: resource stocks and landslide locations, written into DATA (Paper4_Kerala)
import os, io, urllib.request, pandas as pd
os.makedirs(DATA, exist_ok=True)

# Stocks, 4 hubs. Team totals from NDRF reports (55 teams on 18 Aug 2018; 9 teams Aug 2019).
# Hubs, boats (2 per team) and medical units are ASSUMED (stated in the paper; Table 6 S2 tests 25% fewer).
STOCKS = {
 '2018': """base,resource,count
Thrissur,team,14
Thrissur,boat,28
Thrissur,medical,4
Ernakulam,team,14
Ernakulam,boat,28
Ernakulam,medical,4
Thiruvananthapuram,team,14
Thiruvananthapuram,boat,28
Thiruvananthapuram,medical,4
Kozhikode,team,13
Kozhikode,boat,26
Kozhikode,medical,4
""",
 '2019': """base,resource,count
Thrissur,team,3
Thrissur,boat,6
Thrissur,medical,2
Ernakulam,team,2
Ernakulam,boat,4
Ernakulam,medical,2
Thiruvananthapuram,team,2
Thiruvananthapuram,boat,4
Thiruvananthapuram,medical,2
Kozhikode,team,2
Kozhikode,boat,4
Kozhikode,medical,2
"""}

# Hub names must match the district names used as origins in kerala_travel_time_tau0.csv
origins = set(pd.read_csv(DATA + '/kerala_travel_time_tau0.csv')['origin'].astype(str))
print('tau0 origins:', sorted(origins))
for ev, txt in STOCKS.items():
    st = pd.read_csv(io.StringIO(txt))
    fix = {b: next((o for o in origins if o.strip().lower() == b.lower()), None) for b in st['base'].unique()}
    missing = [b for b, o in fix.items() if o is None]
    assert not missing, f'Hub names not found among tau0 origins: {missing}. Rename them to match.'
    st['base'] = st['base'].map(fix)
    st.to_csv(f'{DATA}/stocks_{ev}.csv', index=False)
    print(ev, st.groupby('resource')['count'].sum().to_dict())

# 2019 landslides: the two major sites (8 Aug 2019).
# Kavalappara from landslides.org; Puthumala approximate -- check before final numbers.
pd.DataFrame({'name': ['Kavalappara', 'Puthumala'],
              'lat': [11.410358, 11.5050], 'lon': [76.236181, 76.1500]}).to_csv(DATA + '/landslides_2019.csv', index=False)

# 2018 landslide inventory: Hao et al. (2020) ESSD 12:2899; data van Westen (2020), DANS,
# doi:10.17026/dans-x6c-y7x2, CC BY 4.0. Downloaded from the DANS data station API.
LS = DATA + '/landslides_2018'; os.makedirs(LS, exist_ok=True)
DANS_IDS = {'shp': 6576, 'shx': 6581, 'dbf': 6579, 'prj': 6578, 'cpg': 6575}
for ext, fid in DANS_IDS.items():
    dst = f'{LS}/landslides_2018.{ext}'
    if not os.path.exists(dst) or os.path.getsize(dst) == 0:
        urllib.request.urlretrieve(f'https://phys-techsciences.datastations.nl/api/access/datafile/{fid}', dst)
    print(dst, os.path.getsize(dst), 'bytes')
import geopandas as gpd
lg18 = gpd.read_file(LS + '/landslides_2018.shp')
print('2018 landslides:', len(lg18), lg18.crs, lg18.geom_type.unique(), lg18.total_bounds.round(3))
assert len(lg18) > 4000, 'landslide inventory looks incomplete'


## 1. Pipeline code (identical to `src/` and `reproduce.py` in the repository)

In [ ]:
%%writefile /content/src/common.py
"""Shared imports and constants."""
import math
from dataclasses import dataclass, field, replace
import numpy as np
import pandas as pd
import pulp

CLASSES = ["low", "medium", "high", "severe"]
K = len(CLASSES)
INF = np.inf


In [ ]:
%%writefile /content/src/uncertainty.py
"""uncertainty -- split conformal prediction sets for risk classes (Sect. 6.1)."""
from common import *


def fit_conformal(cal_probs, cal_true, alpha):
    """Split conformal, LAC score 1 - p_true. Returns threshold qhat."""
    n = len(cal_true)
    scores = 1.0 - cal_probs[np.arange(n), cal_true]
    level = min(1.0, math.ceil((n + 1) * (1 - alpha)) / n)
    return float(np.quantile(scores, level, method="higher"))


def conformal_sets(probs, qhat):
    S = (1.0 - probs) <= qhat
    top = probs.argmax(-1)
    np.put_along_axis(S, top[..., None], True, axis=-1)   # never empty
    return S


In [ ]:
%%writefile /content/src/calibration.py
"""calibration -- temperature scaling, calibration metrics (ECE, Brier, NLL, reliability bins) and forecast skill."""
from common import *
from uncertainty import fit_conformal, conformal_sets  # re-exported for convenience


def temperature_scale(probs, T):
    """q = softmax(log p / T). T > 1 softens over-confident forecasts."""
    z = np.log(np.clip(probs, 1e-12, 1.0)) / T
    z = z - z.max(-1, keepdims=True)
    q = np.exp(z)
    return q / q.sum(-1, keepdims=True)


def fit_temperature(cal_probs, cal_true):
    """Fit T by minimising negative log-likelihood on the calibration periods."""
    from scipy.optimize import minimize_scalar
    idx = np.arange(len(cal_true))
    nll = lambda T: -np.log(np.clip(temperature_scale(cal_probs, T)[idx, cal_true], 1e-12, 1)).mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10.0), method="bounded").x)


def calibration_metrics(probs, true, n_bins=10):
    """ECE (top-label), multi-class Brier score, NLL and reliability-diagram bins."""
    probs = probs.reshape(-1, probs.shape[-1])
    true = np.asarray(true).reshape(-1)
    n = len(true)
    conf = probs.max(1)
    pred = probs.argmax(1)
    correct = (pred == true).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    rows, ece = [], 0.0
    for b in range(n_bins):
        m = (conf > edges[b]) & (conf <= edges[b + 1]) if b else (conf >= 0) & (conf <= edges[1])
        if m.sum():
            ece += m.sum() / n * abs(correct[m].mean() - conf[m].mean())
            rows.append({"bin_low": edges[b], "bin_high": edges[b + 1], "count": int(m.sum()),
                         "mean_confidence": conf[m].mean(), "accuracy": correct[m].mean()})
    onehot = np.eye(probs.shape[1])[true]
    return {"ECE": float(ece),
            "Brier": float(((probs - onehot) ** 2).sum(1).mean()),
            "NLL": float(-np.log(np.clip(probs[np.arange(n), true], 1e-12, 1)).mean()),
            "accuracy": float(correct.mean())}, pd.DataFrame(rows)


def forecast_skill(probs, true):
    """Forecasting performance on observed classes: accuracy, macro precision/recall/F1,
    macro one-vs-rest ROC-AUC (classes absent from `true` are skipped)."""
    probs = probs.reshape(-1, probs.shape[-1]); true = np.asarray(true).reshape(-1)
    pred = probs.argmax(1)
    P, R, F, A = [], [], [], []
    for k in range(probs.shape[1]):
        tp = ((pred == k) & (true == k)).sum(); fp = ((pred == k) & (true != k)).sum()
        fn = ((pred != k) & (true == k)).sum()
        if (true == k).sum() == 0:
            continue
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        P.append(p); R.append(r); F.append(2 * p * r / (p + r) if p + r else 0.0)
        pos, neg = probs[true == k, k], probs[true != k, k]
        if len(neg):
            # Mann-Whitney formulation of AUC
            allv = np.concatenate([pos, neg]); ranks = pd.Series(allv).rank().to_numpy()
            A.append((ranks[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))
    return {"accuracy": float((pred == true).mean()), "precision_macro": float(np.mean(P)),
            "recall_macro": float(np.mean(R)), "f1_macro": float(np.mean(F)),
            "roc_auc_ovr_macro": float(np.mean(A)) if A else float("nan")}


In [ ]:
%%writefile /content/src/demand_estimation.py
"""demand_estimation -- part of the CGRR pipeline (Paper 4). Equation numbers refer to the manuscript."""
from common import *


def estimate_delta(cal, cfg):
    d = np.array(cfg.delta_default, float)
    if "exposed_frac" in cal.columns:
        need_mean = cfg.need_beta[0] / sum(cfg.need_beta)
        g = (cal["exposed_frac"] * need_mean).groupby(cal["true_class"]).mean()
        for k, c in enumerate(CLASSES):
            if c in g.index and np.isfinite(g[c]):
                d[k] = g[c]
    return np.maximum.accumulate(d)          # monotone in class


def calibrate_gamma(cal, delta, q):
    probs = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    probs = probs / probs.sum(1, keepdims=True)
    true = cal["true_class"].map(CLASSES.index).to_numpy()
    exceed = delta[true] > probs @ delta
    counts = pd.Series(exceed).groupby(cal["period"].to_numpy()).sum()
    return int(math.ceil(np.quantile(counts, q)))


def bridge(probs, sets, delta, P, v):
    """Eq. (1): demand ranges from conformal sets."""
    vP = (v * P)[None, :]
    big = np.where(sets, delta[None, None, :], -INF).max(-1)
    small = np.where(sets, delta[None, None, :], INF).min(-1)
    return vP * small, vP * (probs @ delta), vP * big   # dL, dhat, dU


In [ ]:
%%writefile /content/src/resource_optimizer.py
"""resource_optimizer -- part of the CGRR pipeline (Paper 4). Equation numbers refer to the manuscript."""
from common import *


def planning_routes(ev, t, cfg, use_hazard=True, learned_closed=None):
    R, J, I = ev.R, ev.J, ev.I
    tau = np.zeros((R, J, I))
    allowed = np.ones((R, J, I), bool)
    for r, name in enumerate(ev.res_names):
        spec = cfg.resources[name]
        base = ev.tau0.copy()
        if use_hazard:
            h = np.broadcast_to(ev.pred_depth[t][None, :], (J, I))
            tau[r] = base * (1 + (cfg.beta * h if spec["mode"] == "road" else 0))
            allowed[r] &= h <= spec["hmax"]
            allowed[r] &= ~ev.known_closed(t, spec["mode"])
        else:
            tau[r] = base
            if learned_closed is not None:
                allowed[r] &= ~learned_closed[spec["mode"]]
        allowed[r] &= np.isfinite(tau[r]) & (tau[r] <= cfg.tau_max)
        if ev.stocks[r].sum() == 0:
            allowed[r] = False
        allowed[r] &= (ev.stocks[r] > 0)[:, None]
    return np.where(np.isfinite(tau), tau, cfg.tau_max * 10), allowed


def _solver(cfg):
    return pulp.HiGHS(msg=False, timeLimit=cfg.solver_time)


def _xvars(allowed, stocks):
    idx = list(zip(*np.nonzero(allowed)))
    return {k: pulp.LpVariable("x_%d_%d_%d" % k, 0, int(stocks[k[0], k[1]]), cat="Integer")
            for k in idx}


def _extract(x, shape):
    out = np.zeros(shape, int)
    for k, var in x.items():
        out[k] = int(round(var.value() or 0))
    return out


def solve_robust(dhat, dU, dL, Gamma, tau, allowed, stocks, kappa, w, vul, xbar, cfg):
    """Model (P), Eqs. (3)-(9). Gamma = 0 and dU = dhat gives the nominal model."""
    R, J, I = allowed.shape
    m = pulp.LpProblem("P", pulp.LpMinimize)
    x = _xvars(allowed, stocks)
    u = [pulp.LpVariable(f"u{i}", 0) for i in range(I)]
    a = [pulp.LpVariable(f"a{i}", 0) for i in range(I)]
    pi = [pulp.LpVariable(f"pi{i}", 0) for i in range(I)]
    th = pulp.LpVariable("theta", 0)
    e = {i: pulp.LpVariable(f"e{i}", 0) for i in np.nonzero(vul)[0]}
    cov = [pulp.lpSum(kappa[r] * x[(r, j, i)] for r in range(R) for j in range(J)
                      if (r, j, i) in x) for i in range(I)]
    obj = (pulp.lpSum(w[i] * u[i] for i in range(I)) + Gamma * th + pulp.lpSum(pi)
           + cfg.lam * pulp.lpSum(tau[k] * x[k] for k in x)
           + cfg.M * pulp.lpSum(w[i] * e[i] for i in e))
    if xbar is not None:
        yp = {k: pulp.LpVariable("yp_%d_%d_%d" % k, 0) for k in x}
        ym = {k: pulp.LpVariable("ym_%d_%d_%d" % k, 0) for k in x}
        for k in x:
            m += x[k] - int(xbar[k]) == yp[k] - ym[k]
        obj += cfg.mu * pulp.lpSum(yp[k] + ym[k] for k in x)
    m += obj
    for i in range(I):
        m += u[i] >= dhat[i] - cov[i]
        m += a[i] >= dU[i] - cov[i]
        m += th + pi[i] >= w[i] * (a[i] - u[i])
    for r in range(R):
        for j in range(J):
            terms = [x[(r, j, i)] for i in range(I) if (r, j, i) in x]
            if terms:
                m += pulp.lpSum(terms) <= int(stocks[r, j])
    for i in e:
        m += cov[i] + e[i] >= cfg.rho * dL[i]
    m.solve(_solver(cfg))
    return _extract(x, (R, J, I))


def solve_stochastic(scen, tau, allowed, stocks, kappa, w, xbar, cfg):
    """B3: minimise expected weighted shortfall over sampled demand scenarios."""
    R, J, I = allowed.shape
    S = scen.shape[0]
    m = pulp.LpProblem("B3", pulp.LpMinimize)
    x = _xvars(allowed, stocks)
    z = [[pulp.LpVariable(f"z{s}_{i}", 0) for i in range(I)] for s in range(S)]
    cov = [pulp.lpSum(kappa[r] * x[(r, j, i)] for r in range(R) for j in range(J)
                      if (r, j, i) in x) for i in range(I)]
    obj = (pulp.lpSum(w[i] * z[s][i] for s in range(S) for i in range(I)) / S
           + cfg.lam * pulp.lpSum(tau[k] * x[k] for k in x))
    if xbar is not None:
        yp = {k: pulp.LpVariable("yp_%d_%d_%d" % k, 0) for k in x}
        ym = {k: pulp.LpVariable("ym_%d_%d_%d" % k, 0) for k in x}
        for k in x:
            m += x[k] - int(xbar[k]) == yp[k] - ym[k]
        obj += cfg.mu * pulp.lpSum(yp[k] + ym[k] for k in x)
    m += obj
    for s in range(S):
        for i in range(I):
            m += z[s][i] >= scen[s, i] - cov[i]
    for r in range(R):
        for j in range(J):
            terms = [x[(r, j, i)] for i in range(I) if (r, j, i) in x]
            if terms:
                m += pulp.lpSum(terms) <= int(stocks[r, j])
    m.solve(_solver(cfg))
    return _extract(x, (R, J, I))


def greedy(score, need, tau, allowed, stocks, kappa):
    """Assign resource units one at a time to the reachable unit with the best score.

    score(i_array, remaining, r, j) -> values for candidate units.
    """
    R, J, I = allowed.shape
    x = np.zeros((R, J, I), int)
    rem = need.astype(float).copy()
    for r in range(R):
        order = np.argsort(tau[r].min(1))
        for j in order:
            for _ in range(int(stocks[r, j])):
                cand = np.nonzero(allowed[r, j])[0]
                if len(cand) == 0:
                    break
                vals = score(cand, rem, r, j)
                i = cand[int(np.argmax(vals))]
                x[r, j, i] += 1
                rem[i] -= kappa[r]
    return x


In [ ]:
%%writefile /content/src/digital_twin.py
"""digital_twin -- part of the CGRR pipeline (Paper 4). Equation numbers refer to the manuscript."""
from common import *
from uncertainty import conformal_sets
from demand_estimation import bridge
from resource_optimizer import planning_routes, solve_robust, solve_stochastic, greedy


@dataclass
class Config:
    alpha: float = 0.10            # conformal miscoverage level
    eta: float = 0.10              # trigger threshold, Eq. (trigger)
    beta: float = 0.5              # travel-time increase per metre of predicted depth
    tau_max: float = 12.0          # h, maximum useful travel time
    lam: float = 0.01              # weight of travel time in objective
    mu: float = 0.05               # weight of relocations in objective
    rho: float = 0.5               # equity floor share of lower demand
    M: float = 1000.0              # penalty of the soft equity constraint
    vuln_quantile: float = 0.75    # units with w above this quantile form I^v
    gamma_quantile: float = 0.95   # quantile used to calibrate Gamma
    fixed_width: float = 0.30      # +-30 % ranges for B4 and A1
    b4_gamma_frac: float = 0.5     # hand-tuned Gamma of B4 as share of units
    n_runs: int = 200
    n_scen_b3: int = 30
    b5_search: int = 60            # random-search candidates for B5
    # Beta(a,b): share of EXPOSED people needing rescue/evacuation/medical help
    # in a period (mean 0.02) -- ASSUMED; calibrate from rescue/evacuation records
    need_beta: tuple = (2.0, 98.0)
    # fraction of TOTAL unit population needing help, per risk class (used if
    # calibration data has no exposed_frac column) -- ASSUMED, justify in paper
    delta_default: tuple = (0.00001, 0.0001, 0.0005, 0.002)
    depth_proxy_m: float = 1.5     # depth proxy if forecasts lack pred_depth_m
    resources: dict = field(default_factory=lambda: {
        # kappa: people served per decision period; hmax: max crossable depth (m)
        "team":    {"kappa": 150, "hmax": 0.3, "mode": "road"},
        "boat":    {"kappa": 60,  "hmax": 99.0, "mode": "boat"},
        "medical": {"kappa": 80,  "hmax": 0.3, "mode": "road"},
    })
    solver_time: int = 30
    seed: int = 42


@dataclass
class Event:
    name: str
    units: list
    bases: list
    res_names: list
    periods: list
    P: np.ndarray          # [I] total population
    w: np.ndarray          # [I] vulnerability weight
    v: np.ndarray          # [I] vulnerability multiplier
    probs: np.ndarray      # [T,I,K] forecast class probabilities
    true_cls: np.ndarray   # [T,I] observed class index
    pred_depth: np.ndarray # [T,I] predicted inundation depth (m)
    exposed: np.ndarray    # [T,I] observed exposed population
    tau0: np.ndarray       # [J,I] normal travel time (h)
    real_tt: dict          # mode -> [T,J,I] realized travel time, inf = impassable
    stocks: np.ndarray     # [R,J]

    @property
    def T(self): return len(self.periods)
    @property
    def I(self): return len(self.units)
    @property
    def J(self): return len(self.bases)
    @property
    def R(self): return len(self.res_names)

    def known_closed(self, t, mode):
        """Closures reported by the previous period (known at decision time)."""
        if t == 0:
            return np.zeros((self.J, self.I), bool)
        return ~np.isfinite(self.real_tt[mode][t - 1])


def load_event(name, forecasts, units, exposure, tau0, route_times, stocks, cfg,
               class_thresholds=(0.01, 0.05, 0.15)):
    """Build an Event from DataFrames (formats in the notebook / README).

    forecasts   : period, datetime, district, p_low..p_severe, [pred_depth_m], [true_class]
    units       : district, total_pop, [w], [v]
    exposure    : date, district, exposed_pop   (from notebook 01; forward-filled)
    tau0        : origin, district, tau0_hours
    route_times : date, origin, district, mode, tt_hours (blank/inf = impassable)
    stocks      : base, resource, count
    """
    f = forecasts.copy()
    f["datetime"] = pd.to_datetime(f["datetime"])
    per = f[["period", "datetime"]].drop_duplicates().sort_values("period")
    periods = list(per["datetime"])
    unit_list = list(units["district"])
    I, T = len(unit_list), len(periods)
    uidx = {u: i for i, u in enumerate(unit_list)}
    pidx = {p: t for t, p in enumerate(per["period"])}

    probs = np.zeros((T, I, K))
    depth = np.full((T, I), np.nan)
    true = np.full((T, I), -1)
    for r in f.itertuples(index=False):
        t, i = pidx[r.period], uidx[r.district]
        probs[t, i] = [getattr(r, f"p_{c}") for c in CLASSES]
        if "pred_depth_m" in f.columns:
            depth[t, i] = r.pred_depth_m
        if "true_class" in f.columns and isinstance(r.true_class, str):
            true[t, i] = CLASSES.index(r.true_class)
    probs = probs / probs.sum(-1, keepdims=True)
    if np.isnan(depth).any():
        print("NOTE: pred_depth_m missing -> depth proxy = "
              f"{cfg.depth_proxy_m} m x P(high or severe). State this in the paper.")
        proxy = cfg.depth_proxy_m * (probs[..., 2] + probs[..., 3])
        depth = np.where(np.isnan(depth), proxy, depth)

    # observed exposure, forward-filled from satellite dates to periods
    ex = exposure.copy()
    ex["date"] = pd.to_datetime(ex["date"])
    exposed = np.zeros((T, I))
    for i, u in enumerate(unit_list):
        s = ex[ex["district"] == u].sort_values("date").set_index("date")["exposed_pop"]
        if len(s):
            exposed[:, i] = s.reindex(periods, method="ffill").fillna(0).to_numpy()
    P = units["total_pop"].to_numpy(float)
    if (true < 0).any():
        frac = exposed / np.maximum(P, 1)[None, :]
        derived = np.digitize(frac, class_thresholds)
        print("NOTE: true_class missing for some rows -> derived from observed "
              f"exposed fraction with thresholds {class_thresholds}.")
        true = np.where(true < 0, derived, true)

    bases = list(dict.fromkeys(stocks["base"]))
    jidx = {b: j for j, b in enumerate(bases)}
    J = len(bases)
    tau = np.full((J, I), INF)
    for r in tau0.itertuples(index=False):
        if r.origin in jidx and r.district in uidx:
            tau[jidx[r.origin], uidx[r.district]] = r.tau0_hours

    rt = route_times.copy()
    rt["date"] = pd.to_datetime(rt["date"])
    rt["tt_hours"] = pd.to_numeric(rt["tt_hours"], errors="coerce").fillna(INF)
    modes = sorted({v["mode"] for v in cfg.resources.values()})
    real = {m: np.repeat(tau[None], T, 0).copy() for m in modes}
    for m in modes:
        sub = rt[rt["mode"] == m]
        for (o, d), g in sub.groupby(["origin", "district"]):
            if o not in jidx or d not in uidx:
                continue
            s = g.sort_values("date").set_index("date")["tt_hours"]
            s = s[~s.index.duplicated(keep="last")]
            vals = s.reindex(periods, method="ffill")
            vals = vals.fillna(tau[jidx[o], uidx[d]]).to_numpy()
            real[m][:, jidx[o], uidx[d]] = vals

    res_names = list(cfg.resources)
    st = np.zeros((len(res_names), J), int)
    for r in stocks.itertuples(index=False):
        st[res_names.index(r.resource), jidx[r.base]] += int(r.count)

    w = units["w"].to_numpy(float) if "w" in units else np.ones(I)
    v = units["v"].to_numpy(float) if "v" in units else np.ones(I)
    return Event(name, unit_list, bases, res_names, periods, P, w, v, probs, true,
                 depth, exposed, tau, real, st)


METHOD_INFO = {
    "B1": "Static deterministic allocation",
    "B2": "Proportional-to-risk heuristic",
    "B3": "Two-stage stochastic programming",
    "B4": "Rolling budgeted robust (fixed budget)",
    "B5": "Learned score-based dispatcher",
    "CGRR": "Proposed",
    "A1": "Fixed-width ranges",
    "A2a": "Re-plan every period",
    "A2b": "Never re-plan",
    "A3": "No inundation/landslide routing",
    "FU": "Forecast-based allocation (uncalibrated)",
    "FC": "Calibration-guided allocation (calibrated, nominal)",
}


@dataclass
class Plan:
    x: np.ndarray          # [T,R,J,I]
    replans: np.ndarray    # [T] bool
    dU: np.ndarray         # [T,I] upper demand used for planning (for exceedance)
    reason: list           # [T] trigger reason or ''


def _sample_scenarios(probs_t, delta, P, v, S, rng):
    cum = probs_t.cumsum(-1)
    u = rng.random((S, probs_t.shape[0], 1))
    cls = (u > cum[None]).sum(-1).clip(max=K - 1)
    return (v * P)[None, :] * delta[cls]


def train_b5(ev, delta, qhat, cfg, rng):
    """B5: fit greedy score weights by random search on outcomes simulated
    from the forecasts only (never sees realized demand or roads)."""
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    best, best_val = None, INF
    cands = [np.array([1.0, 0.0, 0.0])] + [rng.uniform([0, 0, 0], [1, 1, 0.5])
                                            for _ in range(cfg.b5_search - 1)]
    T_train = list(range(0, ev.T, max(1, ev.T // 12)))
    sims = {t: _sample_scenarios(ev.probs[t], delta, ev.P, ev.v, 10, rng) for t in T_train}
    for th in cands:
        val = 0.0
        for t in T_train:
            tau, allowed = planning_routes(ev, t, cfg)
            _, dh, du = bridge(ev.probs[t:t + 1], conformal_sets(ev.probs[t:t + 1], qhat),
                               delta, ev.P, ev.v)
            dhat, spread = dh[0], np.maximum(du[0] - dh[0], 0)
            sc = lambda c, rem, r, j: (th[0] * ev.w[c] * np.maximum(rem[c], 0)
                                       + th[1] * ev.w[c] * np.maximum(spread[c], 0)
                                       - th[2] * tau[r, j, c] * ev.w[c] * dhat[c] / cfg.tau_max)
            x = greedy(sc, dhat, tau, allowed, ev.stocks, kappa)
            cov = np.einsum("r,rji->i", kappa, x)
            val += (ev.w[None] * np.maximum(sims[t] - cov[None], 0)).sum(1).mean()
        if val < best_val:
            best, best_val = th, val
    return best


def plan_method(ev, method, cfg, qhat, delta, Gamma, probs=None, b5_theta=None, rng=None):
    probs = ev.probs if probs is None else probs
    T, R, J, I = ev.T, ev.R, ev.J, ev.I
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    cap = float((kappa[:, None] * ev.stocks).sum())
    sets = conformal_sets(probs, qhat)
    dL, dhat, dU = bridge(probs, sets, delta, ev.P, ev.v)
    if method in ("B4", "A1"):
        dU = dhat * (1 + cfg.fixed_width)
        dL = dhat * (1 - cfg.fixed_width)
    G = {"B4": int(math.ceil(cfg.b4_gamma_frac * I))}.get(method, Gamma)
    vul = ev.w >= np.quantile(ev.w, cfg.vuln_quantile)
    if np.allclose(ev.w, ev.w[0]):
        vul = np.zeros(I, bool)
    # upper class used by the escalation trigger
    upper = sets.cumsum(-1).argmax(-1) if method != "A1" else probs.argmax(-1)

    xs = np.zeros((T, R, J, I), int)
    rep = np.zeros(T, bool)
    reasons = [""] * T
    xbar, ref = None, None
    learned = {m: np.zeros((J, I), bool) for m in ev.real_tt}
    every = method in ("B2", "B3", "B4", "B5", "A2a", "FU", "FC")
    for t in range(T):
        use_hazard = method != "A3"
        if method == "A3" and t > 0:          # learn closures only from own failures
            for r, name in enumerate(ev.res_names):
                mode = cfg.resources[name]["mode"]
                failed = (xs[t - 1, r] > 0) & ~np.isfinite(ev.real_tt[mode][t - 1])
                learned[mode] |= failed
        tau, allowed = planning_routes(ev, t, cfg, use_hazard,
                                       learned if method == "A3" else None)
        # ---- decide whether to (re)plan
        if xbar is None:
            fire, why = True, "initial"
        elif method in ("B1", "A2b"):
            fire, why = False, ""
        elif every:
            fire, why = True, "every period"
        else:
            d_ref = ref["dhat"]
            delta_t = (ev.w * np.abs(dhat[t] - d_ref)).sum() / max((ev.w * d_ref).sum(), cap)
            esc = bool((upper[t] > ref["upper"]).any())
            fail = bool(((xbar > 0) & ~allowed).any())
            fire = delta_t > cfg.eta or esc or fail
            why = ",".join(n for n, f in (("shift", delta_t > cfg.eta), ("escalation", esc),
                                          ("route", fail)) if f)
        if fire:
            if method == "B2":
                sc = lambda c, rem, r, j: ev.w[c] * np.maximum(rem[c], 0) - 1e-6 * tau[r, j, c]
                x = greedy(sc, dhat[t], tau, allowed, ev.stocks, kappa)
            elif method == "B5":
                th = b5_theta
                spread = np.maximum(dU[t] - dhat[t], 0)
                sc = lambda c, rem, r, j: (th[0] * ev.w[c] * np.maximum(rem[c], 0)
                                           + th[1] * ev.w[c] * spread[c]
                                           - th[2] * tau[r, j, c] * ev.w[c] * dhat[t][c] / cfg.tau_max)
                x = greedy(sc, dhat[t], tau, allowed, ev.stocks, kappa)
            elif method == "B3":
                scen = _sample_scenarios(probs[t], delta, ev.P, ev.v, cfg.n_scen_b3, rng)
                x = solve_stochastic(scen, tau, allowed, ev.stocks, kappa, ev.w, xbar, cfg)
            elif method == "B1":
                x = solve_robust(dhat[t], dhat[t], dL[t], 0, tau, allowed, ev.stocks,
                                 kappa, ev.w, vul, None, cfg)
            elif method in ("FU", "FC"):   # nominal model on point forecasts, with relocation cost
                x = solve_robust(dhat[t], dhat[t], dL[t], 0, tau, allowed, ev.stocks,
                                 kappa, ev.w, vul, xbar, cfg)
            else:  # B4, CGRR, A1, A2a, A2b, A3
                x = solve_robust(dhat[t], dU[t], dL[t], G, tau, allowed, ev.stocks,
                                 kappa, ev.w, vul, xbar, cfg)
            xbar = x
            ref = {"dhat": dhat[t].copy(), "upper": upper[t].copy()}
            rep[t], reasons[t] = True, why
        xs[t] = xbar
    return Plan(xs, rep, dU, reasons)


In [ ]:
%%writefile /content/src/evaluation.py
"""evaluation -- part of the CGRR pipeline (Paper 4). Equation numbers refer to the manuscript."""
from common import *
from dataclasses import replace
from scipy.stats import wilcoxon
from calibration import fit_conformal, conformal_sets, fit_temperature, temperature_scale, calibration_metrics, forecast_skill
from demand_estimation import bridge
from demand_estimation import estimate_delta, calibrate_gamma
from digital_twin import plan_method, train_b5, METHOD_INFO


def draw_need(ev, cfg, n_runs, seed):
    rng = np.random.default_rng(seed)
    return rng.beta(*cfg.need_beta, size=(n_runs, ev.I))


def evaluate(ev, plan, cfg, need, demand_mult=1.0, sets=None):
    T, R, J, I = plan.x.shape
    N = need.shape[0]
    kappa = np.array([cfg.resources[r]["kappa"] for r in ev.res_names])
    d = (ev.v * ev.exposed)[None] * need[:, None, :] * demand_mult      # [N,T,I]
    cov = np.zeros((T, I))
    cov_plan = np.einsum("r,trji->ti", kappa, plan.x)
    rt = np.full((T, I), cfg.tau_max)
    failed = np.zeros(T)
    for r, name in enumerate(ev.res_names):
        tt = ev.real_tt[cfg.resources[name]["mode"]]                    # [T,J,I]
        ok = np.isfinite(tt) & (tt <= cfg.tau_max)
        xr = plan.x[:, r]
        cov += kappa[r] * (xr * ok).sum(1)
        failed += (xr * ~ok).sum((1, 2))
        cand = np.where((xr > 0) & ok, tt, INF).min(1)
        rt = np.minimum(rt, np.where(np.isfinite(cand), cand, cfg.tau_max))
    s = np.maximum(d - cov[None], 0)                                    # [N,T,I]
    D = d.sum((1, 2))
    out = {
        "worst_unit_weighted_unmet": (ev.w[None] * s.sum(1)).max(1),
        "total_weighted_unmet": (ev.w[None, None] * s).sum((1, 2)),
        "total_unmet_people": s.sum((1, 2)),
        "coverage_pct": 100 * (1 - s.sum((1, 2)) / np.maximum(D, 1e-9)),
        "mean_response_h": (d * rt[None]).sum((1, 2)) / np.maximum(D, 1e-9),
    }
    flat_rt = np.broadcast_to(rt[None], d.shape).reshape(N, -1)
    flat_d = d.reshape(N, -1)
    p95, w6 = np.zeros(N), np.zeros(N)
    order = np.argsort(flat_rt[0])
    for n in range(N):
        cw = np.cumsum(flat_d[n, order]) / max(flat_d[n].sum(), 1e-9)
        p95[n] = flat_rt[0, order][min(np.searchsorted(cw, 0.95), len(cw) - 1)]
        w6[n] = 100 * flat_d[n][flat_rt[0] <= 6].sum() / max(flat_d[n].sum(), 1e-9)
    out["p95_response_h"] = p95
    out["share_within_6h_pct"] = w6
    severe = (ev.true_cls == K - 1)[None]
    out["missed_high_risk"] = (severe & (s > 0.5 * d) & (d > 0)).sum((1, 2)).astype(float)
    reloc = np.maximum(np.diff(plan.x, axis=0), 0).sum() if T > 1 else 0
    out["relocations"] = np.full(N, float(reloc))
    out["replans"] = np.full(N, float(plan.replans.sum()))
    out["failed_dispatches"] = np.full(N, float(failed.sum()))
    # fairness: Gini coefficient of event-level served share across units
    served = np.minimum(d, cov[None]).sum(1)                             # [N,I]
    share = served / np.maximum(d.sum(1), 1e-9)
    srt = np.sort(share, 1)
    n_u = share.shape[1]
    gini = (2 * (np.arange(1, n_u + 1)[None] * srt).sum(1) / (n_u * np.maximum(srt.sum(1), 1e-9))
            - (n_u + 1) / n_u)
    out["fairness_gini"] = gini
    dispatched = float(np.einsum("r,trji->", kappa, plan.x))
    out["utilization_pct"] = 100 * np.minimum(d, cov[None]).sum((1, 2)) / max(dispatched, 1e-9)
    inadequate = (cov[None] < 0.5 * d) & (d > 0)
    out["uncovered_exposed_pop"] = (inadequate * ev.exposed[None]).sum((1, 2))
    a = np.maximum(plan.dU - cov_plan, 0)
    out["exceedance_rate"] = (s > a[None] + 1e-9).mean((1, 2))
    if sets is not None:
        hit = np.take_along_axis(sets, ev.true_cls[..., None], -1)[..., 0]
        out["set_coverage"] = np.full(N, hit.mean())
        out["mean_set_size"] = np.full(N, sets.sum(-1).mean())
    return pd.DataFrame(out)


METRICS_LOWER_BETTER = {
    "worst_unit_weighted_unmet": True, "total_weighted_unmet": True,
    "total_unmet_people": True, "coverage_pct": False, "mean_response_h": True,
    "p95_response_h": True, "share_within_6h_pct": False, "missed_high_risk": True,
    "relocations": True, "replans": True, "failed_dispatches": True,
    "fairness_gini": True, "utilization_pct": False, "uncovered_exposed_pop": True,
}


def summarize(results, ref="CGRR"):
    """Median [IQR] per method, plus Wilcoxon signed-rank vs ref with Holm correction."""
    rows, tests = [], []
    for m, df in results.items():
        row = {"method": m}
        for c in METRICS_LOWER_BETTER:
            q1, med, q3 = np.percentile(df[c], [25, 50, 75])
            row[c] = med
            row[c + "_iqr"] = f"[{q1:.3g}, {q3:.3g}]"
        rows.append(row)
        if m == ref:
            continue
        for c in METRICS_LOWER_BETTER:
            diff = results[ref][c].to_numpy() - df[c].to_numpy()
            if np.allclose(diff, 0):
                p = 1.0
            elif np.allclose(diff, diff[0]):
                p = 0.0   # deterministic, constant non-zero difference across runs
            else:
                p = wilcoxon(results[ref][c], df[c]).pvalue
            tests.append({"method": m, "metric": c, "p": p,
                          "median_diff_ref_minus_method": float(np.median(diff))})
    tests = pd.DataFrame(tests)
    if len(tests):
        tests["p_holm"] = np.nan
        for c, g in tests.groupby("metric"):
            order = g["p"].sort_values().index
            k = len(order)
            adj, run = [], 0.0
            for rank, idx in enumerate(order):
                run = max(run, min(1.0, (k - rank) * tests.loc[idx, "p"]))
                adj.append((idx, run))
            for idx, val in adj:
                tests.loc[idx, "p_holm"] = val
    return pd.DataFrame(rows), tests


def degrade_probs(probs, true_cls):
    """Stress S3: double the forecast error vector (one-hot(true) - p)."""
    onehot = np.eye(K)[true_cls]
    p = np.clip(2 * probs - onehot, 1e-6, None)
    return p / p.sum(-1, keepdims=True)


def run_event(ev, cal, cfg, methods=None, log=print):
    """Main experiment for one event.

    Raw forecast probabilities -> temperature scaling fitted on the calibration
    periods -> conformal sets on the calibrated probabilities. FU plans on the
    raw probabilities; every other method uses the calibrated ones.
    """
    methods = methods or list(METHOD_INFO)
    rng = np.random.default_rng(cfg.seed)
    raw_cal = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    raw_cal = raw_cal / raw_cal.sum(1, keepdims=True)
    cal_true = cal["true_class"].map(CLASSES.index).to_numpy()
    Tmp = fit_temperature(raw_cal, cal_true)
    q_cal = temperature_scale(raw_cal, Tmp)
    cal_q = cal.copy()
    for k, c in enumerate(CLASSES):
        cal_q[f"p_{c}"] = q_cal[:, k]
    qhat = fit_conformal(q_cal, cal_true, cfg.alpha)
    delta = estimate_delta(cal_q, cfg)
    Gamma = calibrate_gamma(cal_q, delta, cfg.gamma_quantile)
    ev_raw = ev
    ev = replace(ev_raw, probs=temperature_scale(ev_raw.probs, Tmp))
    calib = {}
    for name, p, tr in (("calibration_raw", raw_cal, cal_true), ("calibration_scaled", q_cal, cal_true),
                        ("test_raw", ev_raw.probs, ev.true_cls), ("test_scaled", ev.probs, ev.true_cls)):
        calib[name], bins = calibration_metrics(p, tr)
        calib[name + "_bins"] = bins
    log(f"[{ev.name}] temperature T={Tmp:.3f}  qhat={qhat:.3f}  delta={np.round(delta, 6)}  Gamma={Gamma}")
    log(f"[{ev.name}] test ECE raw={calib['test_raw']['ECE']:.3f} -> scaled={calib['test_scaled']['ECE']:.3f}; "
        f"Brier {calib['test_raw']['Brier']:.3f} -> {calib['test_scaled']['Brier']:.3f}")
    skill = {"raw": forecast_skill(ev_raw.probs, ev.true_cls),
             "scaled": forecast_skill(ev.probs, ev.true_cls)}
    b5 = train_b5(ev, delta, qhat, cfg, rng) if "B5" in methods else None
    need = draw_need(ev, cfg, cfg.n_runs, cfg.seed)
    sets = conformal_sets(ev.probs, qhat)
    # demand-estimation error: forecast point demand vs realized demand (mean over runs)
    realized = ((ev.v * ev.exposed)[None] * need[:, None, :]).mean(0)
    demand_err = {}
    for name, e_ in (("raw", ev_raw), ("scaled", ev)):
        dh = bridge(e_.probs, conformal_sets(e_.probs, qhat), delta, e_.P, e_.v)[1]
        demand_err[name] = {"MAE": float(np.abs(dh - realized).mean()),
                            "RMSE": float(np.sqrt(((dh - realized) ** 2).mean()))}
    plans, res = {}, {}
    for m in methods:
        e_m = ev_raw if m == "FU" else ev
        if m == "FU":   # uncalibrated: conformal threshold refitted on raw probabilities
            q_m = fit_conformal(raw_cal, cal_true, cfg.alpha)
        else:
            q_m = qhat
        plans[m] = plan_method(e_m, m, cfg, q_m, delta, Gamma, b5_theta=b5,
                               rng=np.random.default_rng(cfg.seed + 1))
        res[m] = evaluate(ev, plans[m], cfg, need, sets=sets)
        log(f"  {m:5s} replans={int(plans[m].replans.sum()):3d}  "
            f"coverage={res[m]['coverage_pct'].median():.1f}%")
    main, tests = summarize({m: res[m] for m in methods})
    info = {"temperature": Tmp, "qhat": qhat, "delta": delta, "Gamma": Gamma, "b5_theta": b5,
            "set_coverage": float(res[methods[0]]["set_coverage"].iloc[0]),
            "mean_set_size": float(sets.sum(-1).mean())}
    return {"plans": plans, "results": res, "summary": main, "tests": tests, "info": info,
            "calibration": calib, "skill": skill, "demand_error": demand_err, "need": need, "qhat": qhat, "delta": delta, "Gamma": Gamma,
            "b5": b5, "ev_cal": ev, "temperature": Tmp}


def run_stress(ev, cal, cfg, base, methods=("B1", "B2", "B3", "B4", "B5", "FU", "FC", "CGRR"), log=print):
    out = []
    qhat, delta, Gamma, b5, need = base["qhat"], base["delta"], base["Gamma"], base["b5"], base["need"]
    ev = base["ev_cal"]
    sets = conformal_sets(ev.probs, qhat)
    scen = {}
    # S1: demand 30 % higher than implied (realization only)
    scen["S1"] = {m: evaluate(ev, base["plans"][m], cfg, need, 1.3, sets) for m in methods}
    # S2: 25 % fewer resources
    ev2 = replace(ev, stocks=np.floor(ev.stocks * 0.75).astype(int))
    scen["S2"] = {m: evaluate(ev2, plan_method(ev2, m, cfg, qhat, delta, Gamma, b5_theta=b5,
                                               rng=np.random.default_rng(cfg.seed + 1)),
                              cfg, need, sets=sets) for m in methods}
    # S3: forecast errors doubled, no recalibration
    p3 = degrade_probs(ev.probs, ev.true_cls)
    ev3 = replace(ev, probs=p3)
    scen["S3"] = {m: evaluate(ev3, plan_method(ev3, m, cfg, qhat, delta, Gamma, b5_theta=b5,
                                               rng=np.random.default_rng(cfg.seed + 1)),
                              cfg, need, sets=conformal_sets(p3, qhat)) for m in methods}
    # S4: sudden escalation -- demand in the 3 most exposed units doubles from mid-event,
    # unforeseen by the forecasts (realization only)
    top = np.argsort(-ev.exposed.max(0))[:3]
    mult = np.ones((ev.T, ev.I)); mult[ev.T // 2:, top] = 2.0
    scen["S4"] = {m: evaluate(ev, base["plans"][m], cfg, need, mult[None], sets) for m in methods}
    for s, res in scen.items():
        base_m = [m for m in methods if m != "CGRR"]
        best = min(base_m, key=lambda m: res[m]["total_weighted_unmet"].median())
        c, b = res["CGRR"], res[best]
        out.append({"scenario": s, "best_baseline": best,
                    "unmet_change_pct": 100 * (c["total_unmet_people"].median() / max(b["total_unmet_people"].median(), 1e-9) - 1),
                    "p95_change_pct": 100 * (c["p95_response_h"].median() / max(b["p95_response_h"].median(), 1e-9) - 1),
                    "coverage_change_pp": c["coverage_pct"].median() - b["coverage_pct"].median()})
        log(f"  {s}: best baseline {best}")
    return pd.DataFrame(out), scen


def run_sensitivity(ev, cal, cfg, alphas=(0.05, 0.10, 0.20, 0.30),
                    etas=(0.0, 0.05, 0.10, 0.20, 0.30), log=print):
    rows = []
    need = draw_need(ev, cfg, cfg.n_runs, cfg.seed)
    cal_probs = cal[[f"p_{c}" for c in CLASSES]].to_numpy()
    cal_probs = cal_probs / cal_probs.sum(1, keepdims=True)
    cal_true = cal["true_class"].map(CLASSES.index).to_numpy()
    Tmp = fit_temperature(cal_probs, cal_true)
    cal_probs = temperature_scale(cal_probs, Tmp)
    cal = cal.copy()
    for k, c in enumerate(CLASSES):
        cal[f"p_{c}"] = cal_probs[:, k]
    ev = replace(ev, probs=temperature_scale(ev.probs, Tmp))
    delta = estimate_delta(cal, cfg)
    Gamma = calibrate_gamma(cal, delta, cfg.gamma_quantile)
    for a in alphas:
        qhat = fit_conformal(cal_probs, cal_true, a)
        sets = conformal_sets(ev.probs, qhat)
        for e in etas:
            c2 = replace(cfg, alpha=a, eta=e)
            p = plan_method(ev, "CGRR", c2, qhat, delta, Gamma)
            r = evaluate(ev, p, c2, need, sets=sets)
            rows.append({"alpha": a, "eta": e,
                         "set_coverage": r["set_coverage"].iloc[0],
                         "mean_set_size": r["mean_set_size"].iloc[0],
                         "total_weighted_unmet": r["total_weighted_unmet"].median(),
                         "worst_unit_weighted_unmet": r["worst_unit_weighted_unmet"].median(),
                         "coverage_pct": r["coverage_pct"].median(),
                         "relocations": r["relocations"].iloc[0],
                         "replans": r["replans"].iloc[0],
                         "exceedance_rate": r["exceedance_rate"].median()})
        log(f"  alpha={a} done")
    return pd.DataFrame(rows)


def latex_main_table(summary, methods):
    cols = ["worst_unit_weighted_unmet", "total_unmet_people", "mean_response_h",
            "p95_response_h", "coverage_pct", "missed_high_risk", "relocations"]
    names = {"B1": "B1 Static", "B2": "B2 Proportional", "B3": "B3 Stochastic",
             "B4": "B4 Fixed robust", "B5": "B5 Learned", "FU": "FU Uncalibrated",
             "FC": "FC Calibrated", "CGRR": "CGRR"}
    s = summary.set_index("method")
    lines = []
    for m in methods:
        vals = []
        for c in cols:
            v = s.loc[m, c]
            vals.append(f"{v:,.0f}" if abs(v) >= 100 else f"{v:.2f}")
        lines.append(f"{names.get(m, m)} & " + " & ".join(vals) + r" \\")
    return "\n".join(lines)


def run_demand_sensitivity(ev, cfg, base, means=(0.01, 0.02, 0.05),
                           methods=("B1", "B2", "B3", "B4", "B5", "FU", "FC", "CGRR"), conc=100.0):
    """Alternative demand assumptions: mean share of exposed people needing help.
    Plans are unchanged; only the realized-demand model varies."""
    rows = []
    evc = base["ev_cal"]
    for mval in means:
        c2 = replace(cfg, need_beta=(mval * conc, (1 - mval) * conc))
        need = draw_need(evc, c2, cfg.n_runs, cfg.seed)
        for m in methods:
            r = evaluate(evc, base["plans"][m], c2, need)
            rows.append({"need_mean": mval, "method": m,
                         "total_unmet_people": r["total_unmet_people"].median(),
                         "coverage_pct": r["coverage_pct"].median(),
                         "worst_unit_weighted_unmet": r["worst_unit_weighted_unmet"].median()})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /content/src/preprocessing.py
"""preprocessing -- build realized route travel times per date from the road graph,
flooded road segments (notebook 01) and landslide road blockages.

road mode : flooded and landslide-blocked segments are impassable
boat mode : landslide-blocked segments impassable; flooded segments
            passable at travel_time x boat_slowdown (boats on water)
"""
import numpy as np
import pandas as pd
import networkx as nx


def blocked_edges_from_landslides(edges_gdf, landslides_gdf, buffer_m=50):
    """edge_ids within buffer_m of any landslide point (both in any CRS)."""
    import geopandas as gpd
    e = edges_gdf.to_crs(32643)
    l = landslides_gdf.to_crs(32643)
    j = gpd.sjoin_nearest(l, e[["edge_id", "geometry"]], max_distance=buffer_m, how="inner")
    return set(j["edge_id"].astype(int))


def route_times(G, edge_table, origins, dests, dates, flooded_by_date, blocked_by_date,
                boat_slowdown=3.0, weight="travel_time"):
    """
    G               : networkx (Multi)DiGraph with `weight` in seconds
    edge_table      : DataFrame edge_id, u, v, key
    origins, dests  : dict name -> node
    dates           : list of date strings
    flooded_by_date : dict date -> set(edge_id)
    blocked_by_date : dict date -> set(edge_id)  (landslides active on that date)
    returns DataFrame date, origin, district, mode, tt_hours (inf = impassable)
    """
    et = edge_table.set_index("edge_id")
    Gu = G.to_undirected()
    base_w = {}
    for u, v, k, d in Gu.edges(keys=True, data=True):
        base_w[(u, v, k)] = float(d.get(weight, 1.0))

    def ekeys(ids):
        out = set()
        for e in ids:
            if e in et.index:
                r = et.loc[e]
                out.add((r.u, r.v, r.key))
                out.add((r.v, r.u, r.key))
        return out

    rows = []
    for date in dates:
        fl = ekeys(flooded_by_date.get(date, set()))
        bl = ekeys(blocked_by_date.get(date, set()))
        for mode in ("road", "boat"):
            H = nx.MultiGraph()
            H.add_nodes_from(Gu.nodes)
            for (u, v, k), w in base_w.items():
                if (u, v, k) in bl:
                    continue
                if (u, v, k) in fl:
                    if mode == "road":
                        continue
                    w = w * boat_slowdown
                H.add_edge(u, v, key=k, w=w)
            for o, on in origins.items():
                dist = nx.single_source_dijkstra_path_length(H, on, weight="w")
                for dname, dn in dests.items():
                    tt = dist.get(dn, np.inf)
                    rows.append({"date": date, "origin": o, "district": dname,
                                 "mode": mode, "tt_hours": tt / 3600.0})
    return pd.DataFrame(rows)


In [ ]:
%%writefile /content/src/flood_forecasting.py
"""flood_forecasting -- interface to the forecasting layer.

The contribution of this paper is NOT a new flood prediction model. Any model
that issues, every decision period, class probabilities for each unit can be
plugged in (e.g. the authors' multi-hazard model driven by GPM IMERG rainfall
and GloFAS reforecast discharge). This module only validates its output.
"""
from common import *

REQUIRED = ["period", "datetime", "district"] + [f"p_{c}" for c in CLASSES]


def validate_forecasts(df, name="forecasts"):
    missing = [c for c in REQUIRED if c not in df.columns]
    assert not missing, f"{name}: missing columns {missing}"
    p = df[[f"p_{c}" for c in CLASSES]].to_numpy()
    assert (p >= 0).all(), f"{name}: negative probabilities"
    s = p.sum(1)
    assert np.allclose(s, 1, atol=1e-3), f"{name}: probabilities do not sum to 1 (min {s.min():.3f}, max {s.max():.3f})"
    if "true_class" in df.columns:
        bad = set(df["true_class"].dropna()) - set(CLASSES)
        assert not bad, f"{name}: unknown classes {bad}"
    dup = df.duplicated(["period", "district"]).sum()
    assert dup == 0, f"{name}: {dup} duplicated (period, district) rows"
    return True


def chronological_split_check(cal, test_start):
    """Calibration periods must end before the test event starts (no leakage)."""
    if "datetime" in cal.columns:
        last = pd.to_datetime(cal["datetime"]).max()
        assert last < pd.Timestamp(test_start), f"calibration data ({last}) overlaps test event ({test_start})"
    return True


In [ ]:
%%writefile /content/src/synthetic.py
"""SMOKE-TEST DATA ONLY. Generates random inputs in the exact formats the
pipeline expects, to check that the code runs. Never report these numbers."""
import numpy as np
import pandas as pd
from common import CLASSES


def make(seed=0, I=14, J=5, T=40, n_cal_periods=60):
    rng = np.random.default_rng(seed)
    units = pd.DataFrame({"district": [f"D{i:02d}" for i in range(I)],
                          "total_pop": rng.integers(800_000, 3_500_000, I),
                          "w": rng.uniform(0.5, 1.5, I)})
    units["v"] = 1.0
    dates = pd.date_range("2018-08-08", periods=T, freq="6h")
    # latent severity rises then falls, differs by unit
    peak = rng.integers(10, 30, I)
    sev = np.array([[3.2 * np.exp(-((t - peak[i]) / 9.0) ** 2) for i in range(I)] for t in range(T)])
    noise = np.cumsum(rng.normal(0, 0.06, sev.shape), 0)      # slowly varying forecast error
    sev = sev + noise

    def probs_from(s):
        logits = -((np.arange(4)[None, None] - s[..., None]) ** 2)
        p = np.exp(logits); return p / p.sum(-1, keepdims=True)

    true = np.clip(np.round(sev + rng.normal(0, 0.5, sev.shape)), 0, 3).astype(int)
    pr = probs_from(sev)
    rows = []
    for t in range(T):
        for i in range(I):
            rows.append({"period": t, "datetime": dates[t], "district": units.district[i],
                         **{f"p_{c}": pr[t, i, k] for k, c in enumerate(CLASSES)},
                         "pred_depth_m": max(0, 0.6 * sev[t, i] + rng.normal(0, 0.1)),
                         "true_class": CLASSES[true[t, i]]})
    forecasts = pd.DataFrame(rows)
    # calibration periods (pre-event)
    sc = rng.uniform(0, 3.2, (n_cal_periods, I)) + rng.normal(0, 0.35, (n_cal_periods, I))
    tc = np.clip(np.round(sc + rng.normal(0, 0.5, sc.shape)), 0, 3).astype(int)
    pc = probs_from(sc)
    cal = pd.DataFrame([{"period": p, "district": units.district[i],
                         **{f"p_{c}": pc[p, i, k] for k, c in enumerate(CLASSES)},
                         "true_class": CLASSES[tc[p, i]],
                         "exposed_frac": [0.0005, 0.005, 0.02, 0.06][tc[p, i]] * rng.uniform(0.5, 1.5)}
                        for p in range(n_cal_periods) for i in range(I)])
    # observed exposure on satellite dates (every 4th period)
    frac = np.array([0.0005, 0.005, 0.02, 0.06])[true]
    exp_rows = [{"date": dates[t], "district": units.district[i],
                 "exposed_pop": units.total_pop[i] * frac[t, i] * rng.uniform(0.7, 1.3)}
                for t in range(0, T, 4) for i in range(I)]
    exposure = pd.DataFrame(exp_rows)
    bases = [f"B{j}" for j in range(J)]
    tau0 = pd.DataFrame([{"origin": b, "district": u, "tau0_hours": rng.uniform(0.5, 6)}
                         for b in bases for u in units.district])
    rt = []
    slid = rng.random((J, I)) < 0.08          # landslide blocks persist from period 12
    for t in range(T):
        for b in bases:
            for i, u in enumerate(units.district):
                base = float(tau0[(tau0.origin == b) & (tau0.district == u)].tau0_hours.iloc[0])
                j = bases.index(b)
                flooded = true[t, i] >= 3
                slide = bool(slid[j, i]) and t >= 12
                rt.append({"date": dates[t], "origin": b, "district": u, "mode": "road",
                           "tt_hours": np.inf if (flooded or slide) else base * rng.uniform(1, 1.5)})
                rt.append({"date": dates[t], "origin": b, "district": u, "mode": "boat",
                           "tt_hours": np.inf if slide else base * (2.0 if flooded else 1.0)})
    route_times = pd.DataFrame(rt)
    stocks = pd.DataFrame([{"base": b, "resource": r, "count": c}
                           for b in bases for r, c in (("team", 10), ("boat", 12), ("medical", 4))])
    return dict(forecasts=forecasts, units=units, exposure=exposure, tau0=tau0,
                route_times=route_times, stocks=stocks, calibration=cal)


In [ ]:
%%writefile /content/reproduce.py
"""Reproduce all experiments: python reproduce.py [configs/experiment.yaml]

Writes Tables 3-7 (CSV + LaTeX rows), figures and a run log with input
checksums to results/. With smoke_test: true it uses random data (code check).
"""
import sys, os, json, time, hashlib, platform
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))
import yaml, numpy as np, pandas as pd
import digital_twin as dt, evaluation as evm, synthetic
from flood_forecasting import validate_forecasts

cfg_file = sys.argv[1] if len(sys.argv) > 1 else "configs/experiment.yaml"
C = yaml.safe_load(open(cfg_file))
m, x = C["model"], C["experiment"]
cfg = dt.Config(alpha=m["alpha"], eta=m["eta"], beta=m["beta"], tau_max=m["tau_max"], lam=m["lam"],
                mu=m["mu"], rho=m["rho"], fixed_width=m["fixed_width"], b4_gamma_frac=m["b4_gamma_frac"],
                need_beta=tuple(m["need_beta"]), resources=m["resources"], n_runs=x["n_runs"],
                n_scen_b3=x["n_scen_b3"], b5_search=x["b5_search"], seed=x["seed"])
OUT = C["results_dir"] + ("/SMOKE_TEST" if C["smoke_test"] else "")
for sub in ("tables", "figures", "metrics"):
    os.makedirs(f"{OUT}/{sub}", exist_ok=True)
LOG = []
def log(s):
    line = time.strftime("%Y-%m-%d %H:%M:%S ") + str(s); print(line); LOG.append(line)
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()[:16] if os.path.exists(p) else "missing"

log(f"START smoke_test={C['smoke_test']} python={platform.python_version()} numpy={np.__version__} pandas={pd.__version__}")
log("CONFIG " + json.dumps(C))
D, hashes = C["data_dir"], {}
for k, ev_name in enumerate(C["events"]):
    if C["smoke_test"]:
        d = synthetic.make(seed=k)
        ev = dt.load_event("SMOKE-" + ev_name, d["forecasts"], d["units"], d["exposure"], d["tau0"],
                           d["route_times"], d["stocks"], cfg)
        cal = d["calibration"]
    else:
        files = {"forecasts": f"{D}/forecasts_{ev_name}.csv", "stocks": f"{D}/stocks_{ev_name}.csv",
                 "route_times": f"{D}/route_times_{ev_name}.csv", "exposure": f"{D}/kerala_flood_exposure_all.csv",
                 "tau0": f"{D}/kerala_travel_time_tau0.csv", "calibration": f"{D}/calibration.csv"}
        hashes[ev_name] = {k2: sha(p) for k2, p in files.items()}
        fc = pd.read_csv(files["forecasts"]); validate_forecasts(fc)
        ex = pd.read_csv(files["exposure"]); ex = ex[ex["event"].astype(str) == ev_name]
        ex = ex[ex["imaged_frac"] >= 0.5]   # dates on which Sentinel-1 did not image the district are unobserved
        units = ex.groupby("district", as_index=False)["total_pop"].max()
        vf = f"{D}/vulnerability.csv"
        if os.path.exists(vf):
            units = units.merge(pd.read_csv(vf), on="district", how="left").fillna({"w": 1.0, "v": 1.0})
        ev = dt.load_event(ev_name, fc, units, ex, pd.read_csv(files["tau0"]), pd.read_csv(files["route_times"]),
                           pd.read_csv(files["stocks"]), cfg)
        cal = pd.read_csv(files["calibration"])
    res = evm.run_event(ev, cal, cfg, log=log)
    T = f"{OUT}/tables"
    pd.DataFrame({"raw": res["skill"]["raw"], "temperature_scaled": res["skill"]["scaled"]}).to_csv(f"{T}/table3_forecast_skill_{ev_name}.csv")
    pd.DataFrame({k2: v for k2, v in res["calibration"].items() if not k2.endswith("_bins")}).to_csv(f"{T}/table4_calibration_{ev_name}.csv")
    res["calibration"]["test_raw_bins"].to_csv(f"{OUT}/metrics/reliability_raw_{ev_name}.csv", index=False)
    res["calibration"]["test_scaled_bins"].to_csv(f"{OUT}/metrics/reliability_scaled_{ev_name}.csv", index=False)
    pd.DataFrame(res["demand_error"]).to_csv(f"{T}/table4b_demand_error_{ev_name}.csv")
    main_methods = ["B1", "B2", "B3", "B4", "B5", "FU", "FC", "CGRR"]
    s = res["summary"]
    s[s.method.isin(main_methods)].to_csv(f"{T}/table5_allocation_{ev_name}.csv", index=False)
    s[s.method.isin(["CGRR", "A1", "A2a", "A2b", "A3", "FC"])].to_csv(f"{T}/table7_ablation_{ev_name}.csv", index=False)
    res["tests"].to_csv(f"{OUT}/metrics/wilcoxon_holm_{ev_name}.csv", index=False)
    open(f"{T}/table5_latex_rows_{ev_name}.tex", "w").write(evm.latex_main_table(s, main_methods))
    st, _ = evm.run_stress(ev, cal, cfg, res, log=log)
    se = evm.run_sensitivity(ev, cal, cfg, alphas=x["sensitivity_alpha"], etas=x["sensitivity_eta"], log=log)
    dm = evm.run_demand_sensitivity(ev, cfg, res, means=x["demand_means"])
    st.to_csv(f"{T}/table6_stress_{ev_name}.csv", index=False)
    se.to_csv(f"{T}/table6b_sensitivity_alpha_eta_{ev_name}.csv", index=False)
    dm.to_csv(f"{T}/table6c_demand_assumptions_{ev_name}.csv", index=False)
    p = res["plans"]["CGRR"]
    pd.DataFrame({"period": range(ev.T), "datetime": ev.periods, "replanned": p.replans,
                  "reason": p.reason}).to_csv(f"{OUT}/metrics/trigger_log_{ev_name}.csv", index=False)
    log(f"[{ev_name}] done")
log("END")
with open(f"{OUT}/run_log.txt", "w") as f:
    f.write("\n".join(LOG) + "\nINPUT_SHA256_16 " + json.dumps(hashes) + "\n")
if C["smoke_test"]:
    print("\n*** SMOKE TEST: random data. Do NOT use these numbers in the paper. ***")


## 2. Configuration
Every value marked ASSUMED must be justified in Section 6 of the paper. In particular, `kappa` (people served per unit per 6 h), `hmax` and `need_beta` should come from NDRF/SDMA records or the literature.

In [ ]:
CONFIG = '''# Experiment configuration (Paper 4). Values marked ASSUMED must be justified in Sect. 6.
smoke_test: SMOKE
events: ["2018", "2019"]
data_dir: DATADIR
results_dir: RESULTSDIR

model:
  alpha: 0.10               # conformal miscoverage (target coverage 0.90)
  eta: 0.10                 # trigger threshold
  beta: 0.5                 # travel-time increase per metre of predicted depth   (ASSUMED, Pregnolato et al. 2017)
  tau_max: 12.0             # hours                                              (ASSUMED)
  lam: 0.01                 # travel-time weight
  mu: 0.05                  # relocation weight
  rho: 0.5                  # equity floor
  fixed_width: 0.30         # +-30 % ranges for B4 / A1
  b4_gamma_frac: 0.5        # hand-tuned Gamma of B4
  need_beta: [2.0, 98.0]    # share of exposed people needing help, mean 0.02    (ASSUMED)
  resources:
    team:    {kappa: 150, hmax: 0.3,  mode: road}   # (ASSUMED)
    boat:    {kappa: 60,  hmax: 99.0, mode: boat}   # (ASSUMED)
    medical: {kappa: 80,  hmax: 0.3,  mode: road}   # (ASSUMED)

experiment:
  n_runs: 200
  n_scen_b3: 30
  b5_search: 60
  seed: 42
  sensitivity_alpha: [0.05, 0.10, 0.20, 0.30]
  sensitivity_eta: [0.0, 0.05, 0.10, 0.20, 0.30]
  demand_means: [0.01, 0.02, 0.05]
'''
CONFIG = CONFIG.replace('SMOKE', str(SMOKE_TEST).lower()).replace('DATADIR', DATA).replace('RESULTSDIR', RESULTS)
CONFIG = CONFIG.replace('events: ["2018", "2019"]', 'events: ' + str(EVENTS).replace("'", '"'))
open('/content/experiment.yaml', 'w').write(CONFIG)
print(CONFIG)

## 3. Inputs (real-data mode)

| File in `DATA` | Columns | Source |
|---|---|---|
| `kerala_flood_exposure_all.csv` | event, date, district, exposed_pop, total_pop | notebook 01 (Sentinel-1 or Copernicus GFM flood extent + WorldPop) |
| `kerala_travel_time_tau0.csv` | origin, district, tau0_hours | notebook 01 |
| `forecasts_{event}.csv` | period, datetime, district, p_low, p_medium, p_high, p_severe, [pred_depth_m], [true_class] | your forecasting model (inputs: GPM IMERG rainfall, GloFAS reforecast discharge, terrain), one row per unit every 6 h |
| `calibration.csv` | period, datetime, district, p_low…p_severe, true_class, [exposed_frac] | the same model on **earlier** monsoon periods (chronological split; never the test events) |
| `stocks_{event}.csv` | base, resource (team/boat/medical), count | NDRF records (`data/external/`) |
| `route_times_{event}.csv` | date, origin, district, mode (road/boat), tt_hours | built in step 3b |
| `vulnerability.csv` (optional) | district, w, v | vulnerability index |

`true_class` should come from the observed flood (GFM or Sentinel-1). Observations are the ground truth for evaluating forecasts.

### 3b. Realized route times (flooded roads + landslides)
The 2018 landslide inventory (Hao et al. 2020) has no reliable date for every point, so each landslide counts as blocking from `LANDSLIDE_ACTIVE_FROM`. State this in the paper.

In [ ]:
BUILD_ROUTES = not SMOKE_TEST
LANDSLIDE_FILES = {'2018': DATA + '/landslides_2018/landslides_2018.shp', '2019': DATA + '/landslides_2019.csv'}
LANDSLIDE_ACTIVE_FROM = {'2018': '2018-08-15', '2019': '2019-08-08'}
BASES = {}   # same as notebook 01
if BUILD_ROUTES:
    import sys; sys.path.insert(0, '/content/src')
    import osmnx as ox, geopandas as gpd, pandas as pd, preprocessing
    G = ox.load_graphml(DATA + '/kerala_roads.graphml')
    edges = gpd.read_file(DATA + '/kerala_roads.gpkg')
    dist = gpd.read_file(DATA + '/kerala_districts.geojson').to_crs(4326)
    pts = dist.geometry.representative_point()
    dests = dict(zip(dist['district'], ox.distance.nearest_nodes(G, pts.x, pts.y)))
    origins = dict(dests)
    for b, (lat, lon) in BASES.items():
        origins[b] = ox.distance.nearest_nodes(G, lon, lat)
    flooded = pd.read_csv(DATA + '/kerala_flooded_roads.csv')
    for ev_name in EVENTS:
        fl = flooded[flooded['date'].str.startswith(ev_name)]
        dates = sorted(fl['date'].unique())
        f_by = {d: set(g['edge_id']) for d, g in fl.groupby('date')}
        lf = LANDSLIDE_FILES[ev_name]
        if lf.endswith('.csv'):
            l = pd.read_csv(lf); lg = gpd.GeoDataFrame(l, geometry=gpd.points_from_xy(l.lon, l.lat), crs=4326)
        else:
            lg = gpd.read_file(lf)
        blocked = preprocessing.blocked_edges_from_landslides(edges, lg, buffer_m=50)
        b_by = {d: (blocked if d >= LANDSLIDE_ACTIVE_FROM[ev_name] else set()) for d in dates}
        preprocessing.route_times(G, edges[['edge_id', 'u', 'v', 'key']], origins, dests, dates, f_by, b_by) \
            .to_csv(f'{DATA}/route_times_{ev_name}.csv', index=False)
        print(ev_name, len(dates), 'dates,', len(blocked), 'landslide-blocked segments')

## 4. Run all experiments (about 10–20 min)

In [ ]:
%cd /content
!python reproduce.py /content/experiment.yaml

## 5. Tables

In [ ]:
import pandas as pd, glob, os
OUT = RESULTS + ('/SMOKE_TEST' if SMOKE_TEST else '')
for f in sorted(glob.glob(OUT + '/tables/*.csv')):
    print('\n=====', os.path.basename(f)); df = pd.read_csv(f)
    keep = [c for c in df.columns if not c.endswith('_iqr')]
    display(df[keep].round(3))

## 6. Figures (sans-serif text, markers plus colour, 174 mm wide, 600 dpi TIFF)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})
for ev in EVENTS:
    fig, ax = plt.subplots(1, 3, figsize=(6.85, 2.3))
    for tag, mk in (('raw', 's--'), ('scaled', 'o-')):
        b = pd.read_csv(f'{OUT}/metrics/reliability_{tag}_{ev}.csv')
        ax[0].plot(b.mean_confidence, b.accuracy, mk, label='uncalibrated' if tag == 'raw' else 'temperature-scaled')
    ax[0].plot([0, 1], [0, 1], 'k:', lw=0.8); ax[0].set_xlabel('Confidence'); ax[0].set_ylabel('Accuracy')
    ax[0].legend(fontsize=6); ax[0].set_title('(a) Reliability', fontsize=8)
    s = pd.read_csv(f'{OUT}/tables/table6b_sensitivity_alpha_eta_{ev}.csv')
    g = s[s.eta == 0.10]
    ax[1].plot(1 - g.alpha, g.set_coverage, 'o-'); ax[1].plot([0.6, 1], [0.6, 1], 'k:', lw=0.8)
    ax[1].set_xlabel('Target coverage 1−α'); ax[1].set_ylabel('Empirical coverage'); ax[1].set_title('(b) Coverage', fontsize=8)
    t = pd.read_csv(f'{OUT}/metrics/trigger_log_{ev}.csv', parse_dates=['datetime'])
    for k, (r, mk) in enumerate((('shift', '^'), ('escalation', 's'), ('route', 'x'))):
        m = t.reason.fillna('').str.contains(r)
        ax[2].scatter(t.datetime[m], np.full(m.sum(), k), marker=mk)
    ax[2].set_yticks([0, 1, 2]); ax[2].set_yticklabels(['shift', 'escal.', 'route']); ax[2].set_title('(c) Trigger firings', fontsize=8)
    ax[2].tick_params(axis='x', labelrotation=45, labelsize=6)
    fig.tight_layout(); fig.savefig(f'{OUT}/figures/Fig_calibration_trigger_{ev}.tiff', dpi=600); plt.show()

## 7. Proof of execution

In [ ]:
print(open(OUT + '/run_log.txt').read()[-4000:])
if SMOKE_TEST:
    print('\n*** SMOKE TEST: random data. Do NOT use these numbers in the paper. ***')

## 8. Results archive

In [ ]:
import shutil
zp = shutil.make_archive('/content/Paper4_Kerala_results', 'zip', RESULTS)
shutil.copy(zp, DATA + '/Paper4_Kerala_results.zip')
print('Saved', DATA + '/Paper4_Kerala_results.zip', os.path.getsize(zp), 'bytes')